In [0]:
# Databricks notebook source
# DBTITLE 1,Configuration - Auto-detect source

# Source: Delta table in Volumes
sourcePath = "/Volumes/pharma_catalog/bronze/processed_data/consolidated_delta/member/"
goldTable = "pharma_catalog.gold.member_person_bridge"

print("="*60)
print("CONFIGURATION")
print("="*60)
print(f"Source Path: {sourcePath}")
print(f"Gold Table: {goldTable}")
print("="*60)

In [0]:
# Databricks notebook source
# DBTITLE 1,Install libraries
%pip install recordlinkage

In [0]:
# Databricks notebook source
# DBTITLE 1,Import libraries
from pyspark.sql.functions import date_format, monotonically_increasing_id, sha2, concat_ws, col, row_number, substring, lower, coalesce, upper, trim, regexp_replace
from pyspark.sql.window import Window
import pandas as pd
import recordlinkage
from pyspark.sql.types import StructType,StructField, StringType, IntegerType, LongType

In [0]:
# Databricks notebook source
# DBTITLE 1,Load source data
def LoadSource(sourcePath):
    dfMBPGold = spark.read.format("delta").load(sourcePath)
    windowPartition = Window.partitionBy(col("FileId")).orderBy(col("RecordHash").desc())
    sparkMem_df = dfMBPGold.distinct() \
        .withColumn("RecordHash", sha2(concat_ws("||", *dfMBPGold.columns),256)) \
        .withColumn("RowNumber", row_number().over(windowPartition)) \
        .withColumn("UniqueRecord", concat_ws("-",col("FileID"),col("RowNumber"))) \
        .withColumn("BeneficiaryID", upper(trim(col("BeneficiaryID")))) \
        .withColumn("PlanMemberID", upper(trim(col("PlanMemberID")))) \
        .withColumn("UniquePersonKey", upper(trim(col("UniquePersonKey")))) \
        .withColumn("LastName", upper(trim(col("LastName")))) \
        .withColumn("FirstName", upper(trim(col("FirstName")))) \
        .withColumn("LastInitial", upper(substring(trim(col("LastName")),1,1))) \
        .withColumn("FirstInitial", upper(substring(trim(col("FirstName")),1,1))) \
        .withColumn("DateofBirthFormatted",date_format(trim(col("DateofBirth")),"yyyyMMdd")) \
        .withColumn("PhoneNumber", trim(col("PhoneNumber"))) \
        .withColumn("PhoneNumberFormatted", regexp_replace(trim(col("PhoneNumber")),"[^0-9]","")) \
        .withColumn("PermanentAddressLine1", upper(trim(col("PermanentAddressLine1")))) \
        .select("UniqueRecord","FileLayoutID","FileID","RowNumber","LastName","FirstName","Gender","PhoneNumber","PermanentAddressLine1","DateofBirthFormatted","PlanMemberID","BeneficiaryID","UniquePersonKey","LastInitial","FirstInitial","PhoneNumberFormatted")
    return sparkMem_df

In [0]:
# Databricks notebook source
# DBTITLE 1,Define matching rules
RuleMBIColumns = ["BeneficiaryID","FirstInitial","LastInitial","DateofBirthFormatted","PhoneNumberFormatted","PermanentAddressLine1",11]
RulePMIDColumns = ["PlanMemberID","FirstInitial","LastInitial","DateofBirthFormatted","PhoneNumberFormatted","PermanentAddressLine1",11]
RuleUPKColumns = ["UniquePersonKey","FirstInitial","LastInitial","DateofBirthFormatted","PhoneNumberFormatted","PermanentAddressLine1",11]
RuleOtherColumns = ["DateofBirthFormatted","FirstName", "LastName","PhoneNumberFormatted","PermanentAddressLine1",14]
RulesAll = [RuleMBIColumns, RulePMIDColumns, RuleUPKColumns, RuleOtherColumns]
CompareColumns = ["LastInitial", "FirstInitial", "LastName", "FirstName", "BeneficiaryID", "PlanMemberID", "UniquePersonKey", "DateofBirthFormatted", "PhoneNumberFormatted", "PermanentAddressLine1"]

In [0]:
# Databricks notebook source
# DBTITLE 1,Record linkage - Compare function
def RulesToCompare(rules, pandasMem_df):
    matchesAllRules_df = pd.DataFrame()
    for lst in rules:
        indexer = recordlinkage.Index()
        indexer.block(lst[0])
        candidatesBlock = indexer.index(pandasMem_df)
        print(f"Number of candidates: {len(candidatesBlock)}")
        compareBlock = recordlinkage.Compare()
        threshold = lst[-1]
        for col in CompareColumns:
            compareBlock.exact(col, col, label=str(col))
        features = compareBlock.compute(candidatesBlock, pandasMem_df)
        features[lst[0]] = features[lst[0]].apply(lambda x: x*10)
        matchesRule = features[features[lst[:-1]].sum(axis=1) >= threshold]
        matchesRule_df = matchesRule.index.to_frame()
        matchesAllRules_df = pd.concat([matchesAllRules_df, matchesRule_df])
    return matchesAllRules_df

# COMMAND ----------
# DBTITLE 1,Record linkage - Main linking function
def RunLinking(sparkMem_df):
    pandasMem_df = sparkMem_df.toPandas()
    pandasMem_df = pandasMem_df.set_index("UniqueRecord")
    dfCombined = RulesToCompare(RulesAll, pandasMem_df)
    dfMatchedColumnA = dfCombined.rename(columns={0:"A",1:"B"})
    dfMatchedColumnB = dfCombined.rename(columns={0:"B",1:"A"})
    dfMatched = pd.concat([dfMatchedColumnA,dfMatchedColumnB], ignore_index=True)
    dfMatched.drop_duplicates(inplace=True)
    matchesAll_df = pd.concat([dfMatched[["A","B"]].rename(columns={"A": "Record", "B": "Match"}),
                              dfMatched[["B","A"]].rename(columns={"B": "Record", "A": "Match"})]).reset_index()
    matchesSame_df = matchesAll_df[["Record","Record"]]
    matchesSame_df.columns=['Record', 'Match']
    matchesAll_df = pd.concat([matchesAll_df,matchesSame_df])
    matchesAll_df.drop_duplicates(inplace=True)
    distinctRow = dfMatched.groupby('A').head(1).drop('B', axis=1)
    matchesAll_df = pd.merge(matchesAll_df, distinctRow, how="left",left_on="Record", right_on="A").drop('A', axis=1)
    matchesAll_df = pd.merge(matchesAll_df,pandasMem_df["UniqueRecord"],left_on="Record", right_index=True).rename(columns={"UniqueRecord":"RecordID"})
    matchesAll_df = pd.merge(matchesAll_df,pandasMem_df["UniqueRecord"],left_on="Match", right_index=True).rename(columns={"UniqueRecord":"MatchID"})
    matched_df = matchesAll_df.groupby("RecordID") \
            .agg({"MatchID": lambda x: list(pd.unique(x))}) \
            .reset_index()
    matched_df["MatchID"] = matched_df["MatchID"].apply(lambda x: sorted(x))
    matchesAllModified = matchesAll_df.groupby("RecordID").head(1).drop('MatchID', axis=1).drop('Match', axis=1).drop('index', axis=1).drop('Record', axis=1)
    newDFToMatch = pd.merge(matched_df, matchesAllModified, how="left",left_on="RecordID", right_on="RecordID")
    finalPandas_df = pandasMem_df.merge(newDFToMatch,left_on="UniqueRecord", right_on="RecordID", how="left")
    finalPandas_df['MatchID'] = finalPandas_df['MatchID'].fillna(finalPandas_df['UniqueRecord'])
    return spark.createDataFrame(finalPandas_df.astype(str))

In [0]:
# Databricks notebook source
# DBTITLE 1,Define SQL transformations
finalSQL = """
WITH BISPersonWithIdentifiers AS(
SELECT 
   UniqueRecord,FileLayoutID,FileId,RowNumber,LastName,FirstName,DateofBirthFormatted AS DateOfBirth,Gender,PermanentAddressLine1,PhoneNumber,PlanMemberID,BeneficiaryID,UniquePersonKey
  ,case when instr(MatchID,',')=0 then MatchID else substr(MatchID,2,instr(MatchID,',')) end as MatchID
  ,ROW_NUMBER() OVER(PARTITION BY (case when instr(MatchID,',')=0 then MatchID else substr(MatchID,2,instr(MatchID,',')) end) ORDER BY FileId ASC, RowNumber ASC) AS FirstPersonIdentifier
  ,ROW_NUMBER() OVER(PARTITION BY (case when instr(MatchID,',')=0 then MatchID else substr(MatchID,2,instr(MatchID,',')) end) ORDER BY FileId DESC, RowNumber DESC) AS CurrentPersonIdentifier
FROM BISCompletePersonTable
)
,MemberPersonBridge AS(
SELECT 
   fp.UniqueRecord AS BISInternalPersonID
  ,CASE WHEN cp.CurrentPersonIdentifier = 1 THEN 1 ELSE 0 END AS IsCurrent
  ,cp.UniqueRecord,cp.FileLayoutID,cp.FileId,cp.RowNumber,cp.LastName,cp.FirstName,cp.DateOfBirth,cp.Gender,cp.PermanentAddressLine1,cp.PhoneNumber,cp.PlanMemberID,cp.BeneficiaryID,cp.UniquePersonKey,cp.MatchID
FROM BISPersonWithIdentifiers cp
  LEFT JOIN BISPersonWithIdentifiers fp ON cp.MatchId = fp.MatchId AND fp.FirstPersonIdentifier = 1
)
,MemberPersonBridge_CurrPlanMbr AS(
SELECT 
   BISInternalPersonID,IsCurrent,UniqueRecord,FileLayoutID,FileId,RowNumber,LastName,FirstName,DateOfBirth,Gender,PermanentAddressLine1,PhoneNumber,PlanMemberID,BeneficiaryID
  ,ifnull(nullif(PlanMemberID,'None'),'') AS PlanMemberIdModified
  ,ifnull(nullif(UniquePersonKey,'None'),'') AS UniquePersonKeyModified
  ,UniquePersonKey,MatchID
  ,case when ifnull(PlanMemberID,'None')='None' then null when row_number() over(partition by PlanMemberID order by COALESCE(FileId,0) desc, COALESCE(RowNumber,0) desc) = 1 then 1 else 0 end as IsCurrentPlanMemberID
  ,case when ifnull(UniquePersonKey,'None')='None' then null when row_number() over(partition by UniquePersonKey order by COALESCE(FileId,0) desc, COALESCE(RowNumber,0) desc) = 1 then 1 else 0 end as IsCurrentUniquePersonKey
  ,case when BISInternalPersonID = UniqueRecord then 1 else 0 end AS IsOriginalMemberID 
FROM MemberPersonBridge
)
,PUModPop AS(
SELECT *
    ,CASE WHEN PlanMemberIdModified <> '' THEN 1 ELSE 0 END AS IsPlanMemberIdPopulated
    ,CASE WHEN UniquePersonKeyModified <> '' THEN 1 ELSE 0 END AS IsUniquePersonKeyModifiedPopulated
    ,concat(PlanMemberIdModified,'-',UniquePersonKeyModified) AS PMUP
FROM MemberPersonBridge_CurrPlanMbr
)
,Final AS (
SELECT *
    ,CASE WHEN IsPlanMemberIdPopulated = 1 AND IsUniquePersonKeyModifiedPopulated = 1 THEN 'Fail' ELSE COALESCE(IsCurrentPlanMemberID,IsCurrentUniquePersonKey) END AS IsCurrentPMUP
FROM PUModPop
)
SELECT 
   BISInternalPersonID
  ,IsCurrent
  ,UniqueRecord
  ,CAST(FileLayoutID AS INT) AS FileLayoutID
  ,FileId
  ,LastName
  ,FirstName
  ,DateOfBirth
  ,Gender
  ,PermanentAddressLine1
  ,PhoneNumber
  ,PlanMemberID
  ,BeneficiaryID
  ,UniquePersonKey
  ,sha2(concat_ws('|',IfNull(BISInternalPersonID,''),IfNull(IsCurrent,''),IfNull(UniqueRecord,''),IfNull(CAST(FileLayoutID AS STRING),''),IfNull(CAST(FileId AS STRING),''),IfNull(LastName,''),IfNull(FirstName,''),IfNull(DateOfBirth,''),IfNull(Gender,''),IfNull(PermanentAddressLine1,''),IfNull(PhoneNumber,''),IfNull(PlanMemberID,''),IfNull(BeneficiaryID,''),IfNull(UniquePersonKey,''),IfNull(CAST(IsCurrentPlanMemberID AS STRING),''),IfNull(CAST(IsCurrentUniquePersonKey AS STRING),''),IfNull(CAST(IsOriginalMemberID AS STRING),''),IfNull(PMUP,''),IfNull(CAST(IsCurrentPMUP AS STRING),'')), 256) AS hashKey
  ,IsCurrentPlanMemberID
  ,IsCurrentUniquePersonKey
  ,IsOriginalMemberID
  ,PMUP
  ,CAST(IsCurrentPMUP AS INT) AS IsCurrentPMUP
FROM Final
"""

In [0]:
# Databricks notebook source
# DBTITLE 1,Main execution - Process and write to Gold
print(f"\nProcessing data from: {sourcePath}")
sparkMem_df = LoadSource(sourcePath)
numRows = sparkMem_df.count()
print(f"Found {numRows} records")

if numRows == 0:
    print("No records to process")
elif numRows == 1:
    print("Single record - skipping linkage, processing directly")
    convertedSpark_df = sparkMem_df.withColumn("MatchID", col("UniqueRecord"))
    # Cast numeric columns to proper types (same as multi-record path)
    convertedSpark_df = convertedSpark_df.withColumn("FileID", col("FileID").cast(LongType())).withColumn("RowNumber", col("RowNumber").cast(LongType()))
    
    convertedSpark_df.createOrReplaceTempView("BISCompletePersonTable")
    print("Applying business logic transformations...")
    temp_df = spark.sql(finalSQL)
    
    if temp_df.filter("IsCurrentPMUP IS NULL").count() > 0:
        raise Exception("Failed as records contain both a PlanMemberID and UniquePersonKey")
    else:
        print(f"\nWriting to Gold table: {goldTable}")
        spark.sql(f"DROP TABLE IF EXISTS {goldTable}")
        temp_df.write.format("delta").mode("overwrite").saveAsTable(goldTable)
        print("\n✓ Gold layer processing completed successfully!")
else:
    print("Running record linkage...")
    convertedSpark_df = RunLinking(sparkMem_df)
    convertedSpark_df = convertedSpark_df.withColumn("FileID", col("FileID").cast(LongType())).withColumn("RowNumber", col("RowNumber").cast(LongType()))
    
    convertedSpark_df.createOrReplaceTempView("BISCompletePersonTable")
    print("Applying business logic transformations...")
    temp_df = spark.sql(finalSQL)
    
    if temp_df.filter("IsCurrentPMUP IS NULL").count() > 0:
        raise Exception("Failed as records contain both a PlanMemberID and UniquePersonKey")
    else:
        print(f"\nWriting to Gold table: {goldTable}")
        spark.sql(f"DROP TABLE IF EXISTS {goldTable}")
        temp_df.write.format("delta").mode("overwrite").saveAsTable(goldTable)
        print("\n✓ Gold layer processing completed successfully!")